In [3]:
import subprocess
import sys

packages = [
    "pandas",
    "numpy",
    "scipy",
    "matplotlib",
    "seaborn",
    "ipykernel"
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

Defaulting to user installation because normal site-packages is not writeable


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


Defaulting to user installation because normal site-packages is not writeable


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


Defaulting to user installation because normal site-packages is not writeable


...

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


Defaulting to user installation because normal site-packages is not writeable


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


Defaulting to user installation because normal site-packages is not writeable


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [4]:
# ── BORROWER IDENTITY ──────────────────────────────────────────────
borrower_id = None                  # str: unique user identifier
borrower_name = None                # str: full name
borrower_email = None               # str: email address
# ── INCOME ─────────────────────────────────────────────────────────
monthly_income_verified = None      # float: verified income from payroll
monthly_income_stated = None        # float: income the borrower claimed
income_verification_gap = None      # float: difference between stated vs verified
income_volatility = None            # float: std deviation of monthly income
income_streams_count = None         # int: number of distinct income sources

# ── TRANSACTIONS / CASH FLOW ───────────────────────────────────────
avg_monthly_deposits = None         # float: average total deposits per month
avg_monthly_withdrawals = None      # float: average total withdrawals per month
avg_monthly_net_cash_flow = None    # float: deposits minus withdrawals
overdraft_count_12m = None          # int: overdrafts in last 12 months
overdraft_count_3m = None           # int: overdrafts in last 3 months (recent behavior)
months_of_transaction_history = None # int: how many months of data available (max 24)
nsf_fee_count_12m = None            # int: non-sufficient funds fees in last 12 months
# ── SPENDING BEHAVIOR ──────────────────────────────────────────────
avg_monthly_spending = None         # float: total monthly spend
spending_volatility = None          # float: std deviation of monthly spending
gambling_transactions_count = None  # int: number of gambling-related transactions
gambling_spend_12m = None           # float: total gambling spend in last 12 months

# ── LIABILITIES ────────────────────────────────────────────────────
total_credit_card_balance = None    # float: sum of all credit card balances
total_credit_card_limit = None      # float: sum of all credit card limits
credit_utilization_ratio = None     # float: balance / limit (0.0 to 1.0)
num_credit_cards = None             # int: number of credit cards
total_mortgage_balance = None       # float: outstanding mortgage balance
total_monthly_debt_payments = None  # float: sum of all minimum payments due
debt_to_income_ratio = None         # float: monthly debt payments / monthly income
days_until_next_payment_due = None  # int: urgency of upcoming payment

# ── INVESTMENTS / ASSETS ───────────────────────────────────────────
total_investment_balance = None     # float: total across investment accounts
total_savings_balance = None        # float: total in savings accounts
total_checking_balance = None       # float: total in checking accounts
savings_rate = None                 # float: % of income going to savings
months_of_expenses_in_savings = None # float: financial cushion / runway

# ── PLAID NETWORK INSIGHTS ─────────────────────────────────────────
num_lending_apps_connected = None   # int: # of loan/lending apps connected
num_ewa_apps_connected = None       # int: # of earned wage access apps (e.g. Dave, Earnin)
num_savings_apps_connected = None   # int: # of savings tools connected
total_financial_apps_connected = None # int: total apps on Plaid network

# ── DERIVED RISK FEATURES ──────────────────────────────────────────
plaid_lend_score = None             # int: Plaid's LendScore (1-99)
is_income_verified = None           # bool: whether income was verified via payroll
has_investment_account = None       # bool: whether user has any investments
has_savings_account = None          # bool: whether user has a savings account

# ── TARGET VARIABLE ────────────────────────────────────────────────
defaulted = None                    # int: 1 = defaulted, 0 = repaid (what you're predicting)


In [5]:
tier_1_predictors = [
    'debt_to_income_ratio',          # Most classic default signal
    'credit_utilization_ratio',      # How stretched they are on credit
    'overdraft_count_3m',            # Recent financial stress (3m is more predictive than 12m)
    'monthly_income_verified',       # Ground truth income (not stated)
    'borrow_amount',                 # Loan size relative to their profile
    'avg_monthly_net_cash_flow',     # Can they actually afford repayments?
    'plaid_lend_score',              # Plaid's own model — strong standalone signal
]


In [12]:
# Sample borrower data
import pandas as pd
sample_borrower = {
    'debt_to_income_ratio': 0.42,
    'credit_utilization_ratio': 0.65,
    'overdraft_count_3m': 2,
    'monthly_income_verified': 4200.00,
    'borrow_amount': 5000.00,
    'avg_monthly_net_cash_flow': 620.00,
    'plaid_lend_score': 41,
}

# Create DataFrame
df = pd.DataFrame([sample_borrower])

# View it
df


,debt_to_income_ratio,credit_utilization_ratio,overdraft_count_3m,monthly_income_verified,borrow_amount,avg_monthly_net_cash_flow,plaid_lend_score
0,0.42,0.65,2,4200.0,5000.0,620.0,41


In [7]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "scikit-learn"], check=True)

Defaulting to user installation because normal site-packages is not writeable


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


CompletedProcess(args=['/Library/Developer/CommandLineTools/usr/bin/python3', '-m', 'pip', 'install', 'scikit-learn'], returncode=0)

In [8]:
import sklearn

In [13]:
from sklearn.ensemble import IsolationForest

features = df[tier_1_predictors].fillna(df[tier_1_predictors].median())
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

model = IsolationForest(
    contamination=0.1,   # Assume ~10% of applicants are high risk
    random_state=42
)

df['risk_flag'] = model.fit_predict(X_scaled)
# -1 = anomalous/risky borrower, 1 = normal borrower

df['anomaly_score'] = model.decision_function(X_scaled)
# Lower score = higher risk

NameError: name 'StandardScaler' is not defined